# PostProcessing pyspellchecker v2 — Detection Only (Parcor)

Replacement for the original `PostProcessing pyspellchecker.ipynb`. Differences:

1. **Detection only.** No corrections written back. Each token is classified
   as known/unknown and the suggestion is recorded — but the original sentence
   is preserved unchanged. This avoids the false-positive corrections that
   plagued the previous version.
2. **Parcor only.** Skips Billex and Morph entirely (per scope: Billex
   headwords are too short to spellcheck meaningfully; Morph is small).
3. **Word-level caching.** The same Indonesian token appears thousands of
   times across sentences. The original code re-checked every occurrence;
   this version checks each unique token once and reuses the result.
4. **Strategy switch.** Set `DICT_PATH` at the top to `_strategy_B.json`,
   `_strategy_C.json`, or the original `indonesian_dict.json`. Run once per
   strategy to compare.

## What the output looks like

For each Parcor file, produces a `*_Parcor_spellcheck.csv` with columns:

| Column | Description |
|---|---|
| `kalimat_asal` | original source sentence (unchanged) |
| `kalimat_tujuan` | original target sentence (unchanged) |
| `indonesian_side` | `asal` or `tujuan` — which side was checked |
| `total_tokens` | number of Indonesian tokens in this row |
| `unknown_tokens` | semicolon-separated list of unknown tokens |
| `n_unknown` | count of unknown tokens |
| `unknown_rate` | n_unknown / total_tokens |
| `suggestions` | semicolon-separated `unknown→suggestion` pairs |

Plus a per-dict and corpus-wide aggregate CSV.

The corpus-wide aggregate becomes a new metric for sanity check v2.

## 1. Configuration

In [1]:
import os
import re
import time
import json
import pandas as pd
from pathlib import Path
from collections import Counter
from spellchecker import SpellChecker
import concurrent.futures

# === Choose ONE strategy by uncommenting ===
# DICT_PATH = "../spellCheckDicts/indonesian_dict.json"
DICT_PATH = "../spellCheckDicts/indonesian_dict_strategy_B.json"
# DICT_PATH = "../spellCheckDicts/indonesian_dict_strategy_C.json"

# === Paths ===
SRC_PARCOR_DIR = Path("../Ekstraksi/11. Parallel Corpus - Fixed")
LOOKUP_PATH    = Path("../Ekstraksi/12. Parallel Corpus - Spelling Checker/LookupIsFromIndonesia.csv")

# Detection output goes to a new directory so it never touches the existing
# Spelling Checker directory used for downstream consumption.
strategy_tag = Path(DICT_PATH).stem.replace("indonesian_dict", "").lstrip("_") or "original"
DST_DIR = Path(f"../Ekstraksi/12. Parallel Corpus - Spellcheck Detection/{strategy_tag}")
DST_DIR.mkdir(parents=True, exist_ok=True)

# Performance tuning
MAX_WORKERS = 10   # parallelism for file processing
CACHE_SIZE  = 0   # 0 = unlimited cache, otherwise LRU size

print(f"Dictionary:    {DICT_PATH}")
print(f"Source dir:    {SRC_PARCOR_DIR}")
print(f"Output dir:    {DST_DIR.resolve()}")
print(f"Strategy tag:  {strategy_tag}")

assert os.path.exists(DICT_PATH), f"Dictionary not found: {DICT_PATH}"
assert SRC_PARCOR_DIR.exists(), f"Parcor dir not found: {SRC_PARCOR_DIR}"
assert LOOKUP_PATH.exists(), f"Lookup not found: {LOOKUP_PATH}"

Dictionary:    ../spellCheckDicts/indonesian_dict_strategy_B.json
Source dir:    ..\Ekstraksi\11. Parallel Corpus - Fixed
Output dir:    C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\12. Parallel Corpus - Spellcheck Detection\strategy_B
Strategy tag:  strategy_B


## 2. Load spellchecker + lookup

The spellchecker is a single global object — pyspellchecker isn't easily
multi-process-safe, so all threads will share this one instance. That's fine
for read-only detection (no state mutation).

In [2]:
spell = SpellChecker(language=None)
spell.word_frequency.load_dictionary(DICT_PATH)
print(f"Loaded {len(spell.word_frequency.dictionary):,} words into spellchecker")

lookup_df = pd.read_csv(LOOKUP_PATH)


def get_is_from_indonesia(input_path: str) -> tuple[bool, int]:
    """Returns (success, is_from_indonesia). Same logic as original code."""
    filename = os.path.basename(input_path)
    try:
        file_id = filename.split("_")[0].strip()
    except IndexError:
        return False, -1

    for _, row in lookup_df.iterrows():
        raw_kamus = str(row["kamus"])
        match = re.search(r"\d+", raw_kamus)
        dict_id = match.group() if match else raw_kamus.strip()
        if dict_id == file_id:
            return True, int(row["is_from_indonesia"])
    return False, -1

Loaded 140,863 words into spellchecker


## 3. Word-level cache

The single biggest performance win. The original code calls `spell.unknown()`
+ `spell.correction()` on every token in every sentence. The same token
(`yang`, `dan`, `tidak`) appears thousands of times. Cache the result per
unique lowercase token.

Two cache structures:
- `_known_cache`: `{word: bool}` — is this word in the vocabulary?
- `_correction_cache`: `{word: str | None}` — only filled lazily when needed

In [3]:
_known_cache: dict[str, bool] = {}
_correction_cache: dict[str, str | None] = {}


def is_known(word: str) -> bool:
    """Cached check: is `word` in the spellchecker vocabulary?"""
    if word in _known_cache:
        return _known_cache[word]
    # spell.known returns the subset of input words that ARE known;
    # one call with a single word, but result is cached forever.
    result = len(spell.known([word])) > 0
    _known_cache[word] = result
    return result


def get_correction(word: str) -> str | None:
    """Cached suggestion lookup. None if no suggestion exists or it's same as input."""
    if word in _correction_cache:
        return _correction_cache[word]
    sug = spell.correction(word)
    if sug is None or sug == word:
        sug = None
    _correction_cache[word] = sug
    return sug


# Warm the cache for very common Indonesian function words. These dominate
# token frequency in any Parcor and benefit most from caching.
_warm_words = [
    "yang", "dan", "di", "ke", "dari", "dengan", "untuk", "ada", "ini",
    "itu", "tidak", "akan", "sudah", "sedang", "atau", "tetapi", "juga",
    "saya", "dia", "kami", "kita", "mereka", "karena", "bisa",
]
for w in _warm_words:
    is_known(w)
print(f"Warm cache primed with {len(_warm_words)} common words")

Warm cache primed with 24 common words


## 4. Detection on a single sentence

Token-level: for each Indonesian word, record whether it's known and the
suggestion if not. Output is a structured dict, not a corrected string.

In [4]:
# Require length >= 2: matches the dict_builder's token rule and avoids
# flagging single letters (always unknown, no useful correction).
TOKEN_RE = re.compile(r"[A-Za-z]{2,}")


def detect_sentence(sentence: str) -> dict:
    """
    Returns dict with detection details for one sentence:
      total_tokens, unknown_tokens (list of strings),
      suggestions (list of "unknown→suggestion" pairs).
    """
    if not isinstance(sentence, str):
        return {"total_tokens": 0, "unknown_tokens": [], "suggestions": []}

    tokens = TOKEN_RE.findall(sentence)
    unknown_list = []
    suggestion_list = []

    for original in tokens:
        word = original.lower()
        if not is_known(word):
            unknown_list.append(word)
            sug = get_correction(word)
            if sug:
                suggestion_list.append(f"{word}→{sug}")

    return {
        "total_tokens": len(tokens),
        "unknown_tokens": unknown_list,
        "suggestions": suggestion_list,
    }

## 5. Process one Parcor file

Read CSV, pick the Indonesian column based on direction, run detection, write
audit CSV. Returns per-file summary stats for the corpus aggregate.

In [5]:
def process_file(file_name: str) -> dict:
    input_file  = SRC_PARCOR_DIR / file_name
    output_file = DST_DIR / file_name.replace(".csv", "_spellcheck.csv")

    success, is_from_indonesia = get_is_from_indonesia(str(input_file))
    if not success:
        return {"file": file_name, "status": "skip_no_lookup"}

    try:
        df = pd.read_csv(input_file)
    except Exception as e:
        return {"file": file_name, "status": f"read_error: {e}"}

    if "kalimat_asal" not in df.columns or "kalimat_tujuan" not in df.columns:
        return {"file": file_name, "status": "missing_columns"}

    indonesian_col = "kalimat_asal" if is_from_indonesia == 1 else "kalimat_tujuan"
    side_label = "asal" if is_from_indonesia == 1 else "tujuan"

    rows = []
    total_tokens, total_unknown = 0, 0

    for _, row in df.iterrows():
        sentence = row[indonesian_col]
        det = detect_sentence(sentence)
        rows.append({
            "kalimat_asal":   row.get("kalimat_asal", ""),
            "kalimat_tujuan": row.get("kalimat_tujuan", ""),
            "indonesian_side": side_label,
            "total_tokens": det["total_tokens"],
            "unknown_tokens": ";".join(det["unknown_tokens"]),
            "n_unknown": len(det["unknown_tokens"]),
            "unknown_rate": (
                len(det["unknown_tokens"]) / det["total_tokens"]
                if det["total_tokens"] > 0 else 0.0
            ),
            "suggestions": ";".join(det["suggestions"]),
        })
        total_tokens += det["total_tokens"]
        total_unknown += len(det["unknown_tokens"])

    out_df = pd.DataFrame(rows)
    out_df.to_csv(output_file, index=False)

    return {
        "file": file_name,
        "status": "ok",
        "rows": len(out_df),
        "total_tokens": total_tokens,
        "total_unknown": total_unknown,
        "unknown_rate": total_unknown / total_tokens if total_tokens > 0 else 0.0,
        "indonesian_side": side_label,
    }

## 6. Run the batch

Threading is fine here because the spellchecker is read-only after init and
the caches are append-only. Python's GIL means we don't get true CPU
parallelism, but file I/O is dominant for this workload, so threads still help.

In [6]:
files = sorted([
    f for f in os.listdir(SRC_PARCOR_DIR)
    if f.endswith(".csv") and not f.endswith("_audit.csv")
       and not f.startswith("_")
])
print(f"Found {len(files)} Parcor files to process")
print(f"Cache state: {len(_known_cache):,} known, {len(_correction_cache):,} correction lookups\n")

start = time.time()
summaries = []
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_file, f): f for f in files}
    for count, future in enumerate(concurrent.futures.as_completed(futures), 1):
        result = future.result()
        summaries.append(result)
        elapsed = time.time() - start
        rate = count / elapsed if elapsed > 0 else 0
        eta = (len(files) - count) / rate if rate > 0 else 0
        if result["status"] == "ok":
            print(f"[{count}/{len(files)}] {result['file']}  "
                  f"{result['rows']} rows, {result['unknown_rate']:.1%} unknown  "
                  f"({elapsed:.1f}s elapsed, ETA {eta:.0f}s)")
        else:
            print(f"[{count}/{len(files)}] {result['file']}  STATUS: {result['status']}")

elapsed_total = time.time() - start
print(f"\nDone in {elapsed_total:.1f}s "
      f"({elapsed_total / max(1, len(files)):.1f}s per file avg)")
print(f"Cache final: {len(_known_cache):,} known, {len(_correction_cache):,} correction lookups")

Found 68 Parcor files to process
Cache state: 24 known, 0 correction lookups

[1/68] 12_Parcor.csv  4810 rows, 0.0% unknown  (4.5s elapsed, ETA 298s)
[2/68] 19_Parcor.csv  5579 rows, 0.0% unknown  (4.6s elapsed, ETA 153s)
[3/68] 18_Parcor.csv  2341 rows, 0.0% unknown  (4.6s elapsed, ETA 100s)
[4/68] 16_Parcor.csv  1889 rows, 0.0% unknown  (4.6s elapsed, ETA 74s)
[5/68] 13_Parcor.csv  588 rows, 0.0% unknown  (4.6s elapsed, ETA 58s)
[6/68] 14_Parcor.csv  997 rows, 0.0% unknown  (4.8s elapsed, ETA 50s)
[7/68] 10_Parcor.csv  2942 rows, 0.0% unknown  (5.5s elapsed, ETA 48s)
[8/68] 20_Parcor.csv  1719 rows, 0.0% unknown  (7.9s elapsed, ETA 59s)
[9/68] 23_Parcor.csv  2395 rows, 0.0% unknown  (8.1s elapsed, ETA 53s)
[10/68] 21_Parcor.csv  2019 rows, 0.0% unknown  (8.3s elapsed, ETA 48s)
[11/68] 25_Parcor.csv  195 rows, 0.0% unknown  (8.7s elapsed, ETA 45s)
[12/68] 24_Parcor.csv  3518 rows, 0.0% unknown  (9.7s elapsed, ETA 45s)
[13/68] 27_Parcor.csv  1738 rows, 0.0% unknown  (13.1s elapsed, ETA

## 7. Per-dict summary

In [7]:
ok_summaries = [s for s in summaries if s["status"] == "ok"]
summary_df = pd.DataFrame(ok_summaries)
summary_df["dict_id"] = summary_df["file"].apply(
    lambda f: re.match(r"^(\d+)_", f).group(1) if re.match(r"^(\d+)_", f) else None
)
summary_df = summary_df.sort_values(
    "dict_id", key=lambda s: s.astype(int)
).reset_index(drop=True)

summary_df.to_csv(DST_DIR / "_per_dict_summary.csv", index=False)

print(f"=== Per-dict summary (Strategy: {strategy_tag}) ===")
display_cols = ["dict_id", "rows", "total_tokens", "total_unknown",
                "unknown_rate", "indonesian_side"]
print(summary_df[display_cols].head(20).to_string(index=False))
print(f"\nTotal files OK: {len(summary_df)}")
print(f"Total tokens checked: {summary_df['total_tokens'].sum():,}")
print(f"Total unknown:        {summary_df['total_unknown'].sum():,}")
print(f"Mean unknown rate:    {summary_df['unknown_rate'].mean():.1%}")
print(f"Median unknown rate:  {summary_df['unknown_rate'].median():.1%}")

=== Per-dict summary (Strategy: strategy_B) ===
dict_id  rows  total_tokens  total_unknown  unknown_rate indonesian_side
      1  2914         22573           4806      0.212909          tujuan
      2  2047         19699           2191      0.111224          tujuan
      3   125         34506           9061      0.262592          tujuan
      4  2658         17052              0      0.000000            asal
      5   697          1803            931      0.516362            asal
      8   413          5920           1603      0.270777            asal
      9  1049         10158           1396      0.137429          tujuan
     10  2942         16472              0      0.000000            asal
     11   609         29231           6280      0.214840          tujuan
     12  4810         29933              0      0.000000            asal
     13   588         32387              0      0.000000            asal
     14   997         47216              0      0.000000            asal
   

## 8. Top unknown tokens across the corpus

Which words does this strategy most often flag as unknown? If the top of this
list is full of real Indonesian words, the dictionary has coverage gaps.
If it's full of OCR garbage, the spellchecker is doing its job.

In [8]:
# Aggregate unknown tokens across all files
all_unknown = Counter()
for f in DST_DIR.glob("*_Parcor_spellcheck.csv"):
    try:
        df = pd.read_csv(f)
    except Exception:
        continue
    for cell in df["unknown_tokens"].dropna():
        if isinstance(cell, str) and cell:
            for tok in cell.split(";"):
                if tok:
                    all_unknown[tok] += 1

top_unknown_df = pd.DataFrame(
    all_unknown.most_common(100),
    columns=["token", "occurrences"],
)
top_unknown_df.to_csv(DST_DIR / "_top_unknown_tokens.csv", index=False)

print(f"Total unique unknown tokens: {len(all_unknown):,}")
print(f"\n=== Top 30 unknown tokens (Strategy: {strategy_tag}) ===")
print(top_unknown_df.head(30).to_string(index=False))

Total unique unknown tokens: 67,216

=== Top 30 unknown tokens (Strategy: strategy_B) ===
     token  occurrences
    ltuyuo          316
      tiyo          315
      wonu          290
   pasteyi          242
      waqu          189
  tolakoto          188
    mowali          160
    sababu          159
  reungget          158
     rambl          144
    boyito          142
    tulura          138
      peyi          133
tolowaladu          133
      itoh          133
      daaj          131
   tugobaj          131
     gumai          131
       yio          123
      witu          123
     bitua          119
    amaitu          119
      tomp          110
      lhut          110
      nyoe          109
     moali           89
     maroa           85
        bd           84
    wayata           81
     tapeu           80


## 9. Aggregate stats

Single-line summary suitable for thesis tables and direct comparison
across strategies (just diff this row when running with different DICT_PATH).

In [9]:
aggregate = {
    "strategy": strategy_tag,
    "dict_path": DICT_PATH,
    "dict_size": len(spell.word_frequency.dictionary),
    "files_processed": len(summary_df),
    "total_tokens_checked": int(summary_df["total_tokens"].sum()),
    "total_unknown_tokens": int(summary_df["total_unknown"].sum()),
    "unique_unknown_tokens": len(all_unknown),
    "mean_unknown_rate":   round(float(summary_df["unknown_rate"].mean()), 4),
    "median_unknown_rate": round(float(summary_df["unknown_rate"].median()), 4),
    "p95_unknown_rate":    round(float(summary_df["unknown_rate"].quantile(0.95)), 4),
    "elapsed_seconds": round(elapsed_total, 1),
}

agg_df = pd.DataFrame([aggregate])
agg_df.to_csv(DST_DIR / "_aggregate.csv", index=False)
for k, v in aggregate.items():
    print(f"  {k:<25} {v}")
print(f"\nAll outputs written to: {DST_DIR.resolve()}")

  strategy                  strategy_B
  dict_path                 ../spellCheckDicts/indonesian_dict_strategy_B.json
  dict_size                 140863
  files_processed           68
  total_tokens_checked      1434567
  total_unknown_tokens      102017
  unique_unknown_tokens     67216
  mean_unknown_rate         0.0984
  median_unknown_rate       0.0
  p95_unknown_rate          0.3107
  elapsed_seconds           4368.3

All outputs written to: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\12. Parallel Corpus - Spellcheck Detection\strategy_B


## 10. Comparing strategies

After running this notebook with `DICT_PATH` set to each strategy in turn,
you'll have three sub-directories under `12. Parallel Corpus - Spellcheck Detection/`:
- `original/` — using the original `indonesian_dict.json`
- `strategy_B/` — using the tier-filtered Strategy B dict
- `strategy_C/` — using the cross-dict-validated Strategy C dict

Each contains an `_aggregate.csv` row. To compare:

```python
import pandas as pd
from pathlib import Path
strategies = ["original", "strategy_B", "strategy_C"]
agg_rows = [
    pd.read_csv(Path("../Ekstraksi/12. Parallel Corpus - Spellcheck Detection") / s / "_aggregate.csv")
    for s in strategies
]
pd.concat(agg_rows, ignore_index=True)
```

Expected pattern:
- **Original**: highest dict size → lowest unknown rate but admits noise (real
  errors not flagged because they're in the noisy dict)
- **Strategy B**: tier-filtered → moderate unknown rate, fewer false negatives
- **Strategy C**: cross-validated → highest unknown rate but most reliable
  flags (a flagged word is more likely to be a real error)

The "best" strategy depends on whether you want recall (catch all errors,
including some real-word false positives) or precision (only flag tokens
that are very likely errors).